# Pilota OpAI: rilevabilità v0 / v5 / v8

Impostazioni: **GPU T4 ×2**, **Internet attivo**. Non occorrono Secrets o generazioni.
Il piano è congelato nel commit `d0bccd9`; 400 documenti nuovi, una traiettoria per documento, 1.200 testi. Le 534 fonti usate in precedenza sono escluse. v0 è umano; v5/v8 sono riscritture cumulative di testi umani.

Esegui le celle in ordine. Puoi lasciarle proseguire tutte con Run All. Il notebook riusa le funzioni dei detector dello studio MGTBench, con soglie fisse. La regola di fattibilità è almeno un terzo rilevato per versione e detector, sul totale dei 400 documenti.

Per riprendere in una nuova sessione, carica lo ZIP di output più recente come Input (anche già estratto), imposta `RESUME = "auto"` nella cella 1 ed esegui nuovamente. Le cache sono salvate a ogni testo. Lo ZIP è ricreato anche quando una cella di scoring termina con un errore; se la sessione viene chiusa forzatamente, preserva l'ultimo output disponibile prima dello stop.


In [ ]:
from pathlib import Path
import os, sys, json, shutil, subprocess, hashlib, zipfile

RESUME = ""  # prima esecuzione; per ripresa: "auto" oppure percorso cartella/ZIP negli Input
CODE_REVISION = "064358f6bbbf30648b54fa9d2ca9553cbe915b3a"
REPO = Path("/kaggle/working/.opai_code/Humanization")
RUN = Path("/kaggle/working/opai_detectability_pilot")
if not (REPO / ".git").exists():
    REPO.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(["git", "clone", "--branch", "feature/opai-detectability-pilot",
                    "https://github.com/FrancescoM00/Humanization.git", str(REPO)], check=True)
subprocess.run(["git", "checkout", CODE_REVISION], cwd=REPO, check=True)
print("Codice:", subprocess.check_output(["git", "rev-parse", "HEAD"], cwd=REPO, text=True).strip())


In [ ]:
# Cache e ambiente in una cartella verificata scrivibile, con spazio per i due Falcon.
RUNTIME = None
for candidate in [Path("/kaggle/temp/opai_pilot_runtime"), Path("/kaggle/working/.opai_pilot_runtime")]:
    try:
        candidate.mkdir(parents=True, exist_ok=True)
        probe = candidate / ".write_probe"
        probe.write_text("ok"); probe.unlink()
        if shutil.disk_usage(candidate).free >= 38 * 2**30:
            RUNTIME = candidate; break
    except OSError:
        pass
if RUNTIME is None:
    raise RuntimeError("Servono almeno 38 GiB liberi in una cartella scrivibile per i checkpoint Falcon.")
for name in ["tmp", "huggingface/hub", "matplotlib", "cache"]:
    (RUNTIME / name).mkdir(parents=True, exist_ok=True)
os.environ.update({"HF_HOME":str(RUNTIME/"huggingface"),
    "HF_HUB_CACHE":str(RUNTIME/"huggingface/hub"),
    "HUMANIZATION_HF_WRITE_CACHE":str(RUNTIME/"huggingface/hub"),
    "HF_HUB_DISABLE_XET":"1", "TOKENIZERS_PARALLELISM":"false",
    "MPLCONFIGDIR":str(RUNTIME/"matplotlib"), "XDG_CACHE_HOME":str(RUNTIME/"cache"),
    "TMPDIR":str(RUNTIME/"tmp"), "TMP":str(RUNTIME/"tmp"), "TEMP":str(RUNTIME/"tmp")})
VENV = RUNTIME / "venv"
PYTHON = VENV / "bin/python"
if not PYTHON.exists():
    subprocess.run([sys.executable,"-m","venv","--without-pip","--system-site-packages",str(VENV)],check=True)
requirements = REPO / "requirements-opai-pilot.txt"
stamp = VENV / "requirements.sha256"
digest = hashlib.sha256(requirements.read_bytes()).hexdigest()
if not stamp.exists() or stamp.read_text() != digest:
    # --without-pip evita ensurepip; pip viene ereditato dall'immagine tramite system-site-packages.
    subprocess.run([str(PYTHON),"-m","pip","install","-r",str(requirements)],check=True)
    stamp.write_text(digest)
subprocess.run([str(PYTHON),"-c","import torch; print(torch.__version__); print([torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())]); assert torch.cuda.device_count()>=2, 'Selezionare T4 x2'"],check=True)
print("Cache:", RUNTIME, "GiB liberi:", round(shutil.disk_usage(RUNTIME).free/2**30,1))


In [ ]:
# Ripristino del pannello congelato e, se richiesto, degli score già acquisiti.
BASE = REPO / "results_opai_detectability_pilot"
plan_hash = hashlib.sha256((BASE/"config_pilot.json").read_bytes()).hexdigest()
def compatible(directory):
    p=directory/"config_pilot.json"
    return p.is_file() and hashlib.sha256(p.read_bytes()).hexdigest()==plan_hash and (directory/"panel.jsonl").is_file()

fresh_run = not RUN.exists()
if fresh_run:
    shutil.copytree(BASE,RUN)
if hashlib.sha256((RUN/"config_pilot.json").read_bytes()).hexdigest()!=plan_hash:
    raise RuntimeError("Il run esistente appartiene a un altro piano; scegliere un'altra cartella RUN.")
if RESUME and not (RUN/".checkpoint_restored").exists():
    location=Path(RESUME) if RESUME!="auto" else Path("/kaggle/input")
    if location.is_file():
        destination=RUNTIME/"resume_extracted"
        destination.mkdir(exist_ok=True)
        with zipfile.ZipFile(location) as z:
            if any(not (destination/m.filename).resolve().is_relative_to(destination.resolve()) for m in z.infolist()):
                raise RuntimeError("Percorso non valido nello ZIP")
            z.extractall(destination)
        location=destination
    candidates=[p.parent for p in location.rglob("config_pilot.json") if compatible(p.parent)]
    if compatible(location): candidates.append(location)
    candidates=list(dict.fromkeys(candidates))
    if len(candidates)!=1:
        raise RuntimeError(f"Trovati {len(candidates)} checkpoint compatibili. Indica esplicitamente la cartella estratta in RESUME.")
    if not fresh_run and any(RUN.glob("scores_*.csv")):
        raise RuntimeError("Sono già presenti score nel RUN: non sovrascriverli con un checkpoint. Riprendere con RESUME vuoto.")
    shutil.copytree(candidates[0],RUN,dirs_exist_ok=True)
    (RUN/".checkpoint_restored").write_text(str(candidates[0]))

ARCHIVE=Path("/kaggle/working/opai_detectability_pilot.zip")
def export_zip():
    with zipfile.ZipFile(ARCHIVE,"w",compression=zipfile.ZIP_DEFLATED) as z:
        for p in RUN.rglob("*"):
            if p.is_file():z.write(p,Path(RUN.name)/p.relative_to(RUN))
    print("ZIP aggiornato:",ARCHIVE)
def run_stage(*args):
    subprocess.run([str(PYTHON),str(REPO/"scripts/opai_detectability_pilot.py"),
                    "--output",str(RUN),*map(str,args)],cwd=REPO,env=dict(os.environ),check=True)
run_stage("--stage","summarize")
export_zip()


## OpenAI RoBERTa-base

Scoring dei 1.200 testi. La soglia rimane quella calibrata su MGTBench. Il CSV completo, se già presente dopo una ripresa, viene verificato dalla fase di riepilogo e riutilizzato.


In [ ]:
try:
    if not (RUN/"scores_openai_roberta_base.csv").exists():
        run_stage("--stage","score","--only","openai_roberta_base","--device","cuda:0")
    else:
        print("OpenAI già completato: riuso gli score salvati.")
    run_stage("--stage","summarize")
finally:
    export_zip()


## Binoculars Falcon-7B

Un modello su ciascuna T4, FP16 e stesse finestre dello studio originale. Il primo avvio scarica circa 28 GB di pesi; la durata dipende anche dalla rete. Le cache per testo permettono la ripresa. Non usare quantizzazione o sostituire modelli: cambierebbe la condizione a cui si riferisce la soglia.


In [ ]:
try:
    if not (RUN/"scores_binoculars_falcon7b.csv").exists():
        run_stage("--stage","score","--only","binoculars_falcon7b",
                  "--observer-device","cuda:0","--performer-device","cuda:1")
    else:
        print("Binoculars già completato: riuso gli score salvati.")
    run_stage("--stage","summarize")
finally:
    export_zip()


## Risultati da restituire

Scarica `opai_detectability_pilot.zip` dagli Output. Lo stato corretto a fine pilota è `completed`, con **2.400 score**. `summary_rates.csv` contiene conteggi k/n, Wilson e mediane di parole, inclusi i riferimenti MGTBench. `auroc.csv` separa v0/v5 e v0/v8; `feasibility.csv` applica la regola di un terzo senza ricalibrare. Uno stato parziale non va interpretato come zero rilevazioni.


In [ ]:
import pandas as pd
from IPython.display import display, Image, FileLink
run_stage("--stage","summarize")
status=json.loads((RUN/"experiment.json").read_text())
print(json.dumps(status,indent=2))
display(pd.read_csv(RUN/"summary_rates.csv"))
display(pd.read_csv(RUN/"auroc.csv"))
display(pd.read_csv(RUN/"feasibility.csv"))
display(Image(filename=str(RUN/"detection_rates.png")))
export_zip()
display(FileLink(str(ARCHIVE)))
